# Preview: GT skeletonization on full Dataset501_ARCADE images

Quality check for `skimage.morphology.skeletonize` on the **original full-size
(512x512) syntax GT masks** (`Dataset501_ARCADE/labelsTr`, multi-class:
0=background, 1=LAD, 2=RCA, 3=LCX) -- not the 6x6-tiled patches used
elsewhere. This is the same skeletonize call used throughout
`vessel_gap_transform.py` / `prepare_arcade_vessel_coverage.py` /
`preview_smallenet_results.ipynb`'s hypothesis-test section; this notebook
just isolates it on a few full images to visually judge quality (breaks,
spurious spurs, thick-vessel artifacts) before trusting it elsewhere.

Classes are binarized (any class > 0 = vessel) before skeletonizing, since
everywhere else in this repo treats the full tree as one binary mask.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage.morphology import skeletonize


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "enet").exists() and (candidate / "data").exists():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "analysis" / "501_ARCADE"))
import segmentation_topology as topo  # noqa: E402

DATASET_DIR = REPO_ROOT / "data" / "nnUNet_raw" / "Dataset501_ARCADE"
IMAGES_TR_DIR = DATASET_DIR / "imagesTr"
LABELS_TR_DIR = DATASET_DIR / "labelsTr"

print("Repo root  :", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)

## Pick a few masks with substantial, varied vessel content

RCA and LAD/LCX never co-occur in one image (per `segmentation_topology.py`'s
territory logic), so most cases are single-class; a couple of 2-class
(LAD+LCX) cases are included too for a slightly more complex tree shape.

In [ ]:
case_ids = ["train_183", "train_636", "train_130", "train_453"]

for case_id in case_ids:
    arr = topo.load_class_id_mask(LABELS_TR_DIR / f"{case_id}.png")
    classes = sorted(int(c) for c in np.unique(arr) if c != 0)
    print(f"{case_id}: classes present = {classes}, vessel px = {int((arr > 0).sum())}")

## Skeletonize + overlay

In [ ]:
def skeleton_degree(skel: np.ndarray) -> np.ndarray:
    """Neighbor count per skeleton pixel -- 1=tip, 2=body, >=3=branch point.
    Same convolution trick used in vessel_gap_transform.py."""
    kernel = np.ones((3, 3), dtype=int)
    neighbor_count = ndi.convolve(skel.astype(int), kernel, mode="constant") - skel.astype(int)
    return neighbor_count * skel


def dilate_for_display(mask: np.ndarray, radius: int = 1) -> np.ndarray:
    """1px skeleton lines are near-invisible at 512x512 in a small figure --
    dilate purely for display, not used for any measurement."""
    return ndi.binary_dilation(mask, iterations=radius)


fig, axes = plt.subplots(len(case_ids), 4, figsize=(16, 4 * len(case_ids)))
axes = np.atleast_2d(axes)
for col, title in enumerate(["Raw", "GT mask (binary)", "Skeleton overlay", "Skeleton: tips (red) / branch pts (blue)"]):
    axes[0, col].set_title(title, fontsize=10, fontweight="bold")

for row, case_id in enumerate(case_ids):
    raw = np.asarray(Image.open(IMAGES_TR_DIR / f"{case_id}_0000.png").convert("L"))
    class_mask = topo.load_class_id_mask(LABELS_TR_DIR / f"{case_id}.png")
    mask = class_mask > 0
    skel = skeletonize(mask)
    deg = skeleton_degree(skel)
    tips = deg == 1
    branches = deg >= 3

    axes[row, 0].imshow(raw, cmap="gray")
    axes[row, 0].set_ylabel(case_id, fontsize=9, rotation=0, labelpad=40, va="center")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(mask, cmap="gray")
    skel_rgba = np.zeros((*skel.shape, 4))
    skel_rgba[dilate_for_display(skel)] = [1, 0.2, 0.2, 1]  # red
    axes[row, 2].imshow(skel_rgba)
    axes[row, 2].axis("off")

    axes[row, 3].imshow(mask, cmap="gray", alpha=0.4)
    skel_only = np.zeros((*skel.shape, 4))
    skel_only[dilate_for_display(skel & ~tips & ~branches)] = [1, 1, 1, 1]  # white = plain body
    skel_only[dilate_for_display(tips, radius=3)] = [1, 0, 0, 1]  # red = endpoints
    skel_only[dilate_for_display(branches, radius=3)] = [0.2, 0.5, 1, 1]  # blue = branch points
    axes[row, 3].imshow(skel_only)
    axes[row, 3].axis("off")

    n_tips, n_branches = int(tips.sum()), int(branches.sum())
    print(f"{case_id}: skeleton px={int(skel.sum())}  tips={n_tips}  branch points={n_branches}")

plt.tight_layout()
plt.savefig(REPO_ROOT / "analysis" / "501_ARCADE" / "results" / "skeletonization_preview.png", dpi=150, bbox_inches="tight")
plt.show()